In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

class RobotServoDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_paths = sorted([os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.endswith(".png")])
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        base_name = os.path.basename(path).replace(".png", "")
        servo_positions = torch.tensor([float(x)/1000.0 for x in base_name.split("_")], dtype=torch.float32)

        return image, servo_positions

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

dataset = RobotServoDataset("/content/drive/MyDrive/RobotArm", transform=transform)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

class RobotModel(nn.Module):
    def __init__(self):
        super(RobotModel, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1))
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 6)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

model = RobotModel().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

epochs = 100
best_loss = float("inf")

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for x_batch, y_batch in tqdm(train_loader, desc="Train"):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = model(x_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for x_batch, y_batch in tqdm(test_loader, desc="Test"):
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            pred = model(x_batch)
            loss = criterion(pred, y_batch)
            test_loss += loss.item()
    test_loss /= len(test_loader)

    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f}")

    if test_loss < best_loss:
        torch.save(model.state_dict(), "/content/drive/MyDrive/RobotArm/best_robot_model.pth")
        best_loss = test_loss
        print(f"Saved best model at epoch {epoch+1}")


Test: 100%|██████████| 1/1 [00:00<00:00,  7.59it/s]


Epoch 1 | Train Loss: 0.2474 | Test Loss: 0.2405
Saved best model at epoch 1


Test: 100%|██████████| 1/1 [00:00<00:00,  8.20it/s]


Epoch 2 | Train Loss: 0.2446 | Test Loss: 0.2378
Saved best model at epoch 2


Test: 100%|██████████| 1/1 [00:00<00:00,  8.21it/s]


...

Epoch 98 | Train Loss: 0.0078 | Test Loss: 0.0075


Test: 100%|██████████| 1/1 [00:00<00:00,  5.86it/s]


Epoch 99 | Train Loss: 0.0078 | Test Loss: 0.0075


Test: 100%|██████████| 1/1 [00:00<00:00,  5.85it/s]

Epoch 100 | Train Loss: 0.0078 | Test Loss: 0.0074
